# 2DTFIM 2DNQS - (Nx,Ny)=(12,12): Inference (seed 111)

This is part of the work arxiv: 2606.25600 (Two-dimensional Hyperbolic RNN Neural Quantum State). For the purpose of reproducing the results, please check the link to the trained weight files. The saved weight links used in this notebook might not be the same as the ones in the Github repo. 

In [1]:
import sys
import os
sys.path.append('../../../utility_tfim')
from xdrnn_tfim2d_train_loop import *
import time

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=3.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped 

def define_load_test(wf, numsamples,path_to_weights, Ee, clipped_e = False):
    test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_before, wf)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')

     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_after, wf)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=3.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_after, wf)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy (not exact in 2D) is {np.round(Ee,4)}')

In [4]:
Nx=12
Ny=12
units =60
Jz=np.ones((Nx,Ny))
nsamples = 10000
seed=111
set_cpu_deterministic(seed)

# B=2.0

In [9]:
Bx=2.0
fname = f'../2DTFIM_2dNQS_res/(12, 12)_B={Bx}'
E_dmrg=-346.98277872

In [11]:
import glob
# Search recursively for that specific filename
matches = glob.glob("../2DTFIM_2dNQS_res/(12, 12)_B=2.0/**/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed=111_checkpoint.pt", recursive=True)
if matches:
    print(f"File found at: {matches}")
else:
    print("File is nowhere to be found in any subfolder.")

File found at: ['../2DTFIM_2dNQS_res/(12, 12)_B=2.0/Euclidean/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed=111_checkpoint.pt']


### Euclidean 2DRNN

In [6]:
wf = EuclideanRNNwavefunction(Nx, Ny,  units, 2, seed=seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed=111_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -288.6633, var E = 265.8201
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -346.85, var E = 0.7652
DMRG energy (not exact in 2D) is -346.9828
Time taken =0.19 hrs


### Lorentz 2DRNN (L_max=2.0, 6.0)

In [8]:
sc=2.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = 111)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=2.0_seed=111_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -288.4492, var E = 271.9184
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -346.3433, var E = 3.3429
DMRG energy (not exact in 2D) is -346.9828
Time taken =1.765 hrs


In [13]:
sc=6.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax={sc}_seed=111_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -288.7969, var E = 273.3227
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -346.329, var E = 3.4868
DMRG energy (not exact in 2D) is -346.9828
Time taken =2.571 hrs


# B=3.0

In [6]:
Bx=3.0
fname = f'../2DTFIM_2dNQS_res/(12, 12)_B={Bx}'
E_dmrg = -456.9066548624153

### Euclidean 2DRNN

In [6]:
wf = EuclideanRNNwavefunction(Nx, Ny,  units, 2, seed=seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/Euclidean2dRNN_u=60_12x12_ns=80_seed=111_rmax=None_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -432.1826, var E = 263.53
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -456.88, var E = 1.2915
DMRG energy (not exact in 2D) is -456.9067
Time taken =0.181 hrs


### Lorentz 2DRNN (L_max=2.0, 5.0)

In [15]:
sc=2.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = 111)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=2.0_seed=111_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -432.0808, var E = 270.738
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -456.9595, var E = 0.7021
DMRG energy (not exact in 2D) is -456.9067
Time taken =2.706 hrs


In [7]:
sc=5.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = 111)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=5.0_seed=111_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -432.3172, var E = 270.9191
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -456.967, var E = 0.7707
DMRG energy (not exact in 2D) is -456.9067
Time taken =1.9 hrs


# B=4.0

In [11]:
Bx=4.0
E_dmrg=593.53890192
fname = f'../2DTFIM_2dNQS_res/(12, 12)_B={Bx}'

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


### Euclidean 2DRNN

In [10]:
wf = EuclideanRNNwavefunction(Nx, Ny,  units, 2, seed=seed)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/Euclidean2dRNN_u=60_12x12_ns=80_rmax=None_seed=111_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -575.7018, var E = 263.1912
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -593.18, var E = 3.986
DMRG energy (not exact in 2D) is 593.5389
Time taken =0.183 hrs


### Lorentz 2DRNN (L_max=2.0, 1.5)

In [12]:
sc=2.0 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = 111)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=2.0_seed=111_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -575.9453, var E = 270.6006
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -592.9785, var E = 6.9397
DMRG energy (not exact in 2D) is 593.5389
Time taken =2.476 hrs


In [8]:
sc=1.5 
wf= LorentzRNNwavefunction(Nx, Ny, units,  2, 
                        spatial_clamp=sc, non_lin='elu', seed = 111)
wf.model.double()
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Lorentz/Lorentz2dRNN_u=60_12x12_ns=80_elu_Lmax=1.5_seed=111_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 7,622
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -575.9768, var E = 270.0492
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -593.4642, var E = 0.8502
DMRG energy (not exact in 2D) is 593.5389
Time taken =2.118 hrs
